In [0]:
CREATE CATALOG IF NOT EXISTS Legacy_pty;
USE CATALOG Legacy_pty;

CREATE SCHEMA IF NOT EXISTS Bronze;
USE SCHEMA Bronze;

CREATE SCHEMA IF NOT EXISTS Silver;
USE SCHEMA Silver;

CREATE SCHEMA IF NOT EXISTS Gold;
USE SCHEMA Gold;

In [0]:
-- Silver Layer: Cleaned Accounts Data
-- Data cleaning operations: Fix typos, trim whitespace, standardize text, create index-based aliases
-- Business Purpose: Customer dimension table for sales analysis and account hierarchy tracking

CREATE OR REPLACE TABLE Legacy_pty.Silver.accounts AS
WITH indexed_accounts AS (
  SELECT 
    -- Generate surrogate key for stable joins (account names can change)
    -- ROW_NUMBER ensures each account gets unique ID 1-85
    ROW_NUMBER() OVER (ORDER BY sector, account) as account_id,
    account,
    sector,
    year_established,
    revenue,
    employees,
    office_location,
    subsidiary_of
  FROM Legacy_pty.Bronze.accounts
  WHERE account IS NOT NULL  -- Filter out invalid records
)
SELECT 
  account_id,
  TRIM(account) as account,  -- Remove leading/trailing whitespace
  
  -- Data Quality Fix: Standardize sector values to lowercase
  -- Corrects 'technolgy' typo found in source data
  CASE 
    WHEN LOWER(TRIM(sector)) = 'technolgy' THEN 'technology'
    ELSE LOWER(TRIM(sector))
  END as sector,
  
  year_established,
  revenue as company_revenue,  -- Renamed for clarity (company revenue vs. sales revenue)
  employees,
  TRIM(office_location) as office_location,
  
  -- subsidiary_of enables parent-child hierarchy analysis
  -- NULL = parent company, Non-NULL = subsidiary
  TRIM(subsidiary_of) as subsidiary_of,
  
  current_timestamp() as cleaned_at  -- Audit timestamp
FROM indexed_accounts;

In [0]:
-- Silver Layer: Cleaned Products Data
-- Data cleaning operations: Trim whitespace, standardize text, add product_id index

CREATE OR REPLACE TABLE Legacy_pty.Silver.products AS
WITH indexed_products AS (
  SELECT 
    ROW_NUMBER() OVER (ORDER BY series, product) as product_id,
    product,
    series,
    sales_price
  FROM Legacy_pty.Bronze.products
  WHERE product IS NOT NULL
    AND sales_price IS NOT NULL
    AND sales_price > 0
)
SELECT 
  product_id,
  TRIM(product) as product,
  TRIM(series) as series,
  sales_price,
  current_timestamp() as cleaned_at
FROM indexed_products;

In [0]:
-- Silver Layer: Cleaned Sales Pipeline Data
-- Data cleaning operations: Trim whitespace, standardize text, calculate business metrics
-- Business Purpose: Central fact table containing all sales opportunities and their lifecycle

CREATE OR REPLACE TABLE Legacy_pty.Silver.sales_pipeline AS
SELECT 
  -- Primary key for each sales opportunity
  TRIM(opportunity_id) as opportunity_id,
  
  -- Foreign keys to dimension tables (accounts, products, sales_teams)
  TRIM(sales_agent) as sales_agent,
  TRIM(product) as product,
  TRIM(account) as account,
  
  -- Deal lifecycle stage: Prospecting, Engaging, Won, Lost
  -- Won/Lost are terminal states; others are active pipeline
  TRIM(deal_stage) as deal_stage,
  
  -- Date fields for time-series analysis and sales velocity metrics
  engage_date,  -- When sales agent first contacted prospect
  close_date,   -- When deal was Won or Lost (NULL for active deals)
  
  -- Revenue field: only realized when deal_stage = 'Won'
  -- For reporting, filter WHERE deal_stage = 'Won' to get actual revenue
  close_value,
  
  -- CALCULATED METRIC: Sales cycle length (time from engagement to close)
  -- Critical for forecasting and capacity planning
  -- NULL for active deals (close_date not yet set)
  DATEDIFF(close_date, engage_date) as days_to_close,
  
  current_timestamp() as cleaned_at  -- Data lineage timestamp
FROM Legacy_pty.Bronze.sales_pipeline
WHERE opportunity_id IS NOT NULL    -- Required: unique identifier
  AND sales_agent IS NOT NULL       -- Required: must have owner
  AND engage_date IS NOT NULL;      -- Required: deal must be started
-- Note: close_date and close_value can be NULL for active deals

In [0]:
-- Silver Layer: Cleaned Sales Teams Data
-- Data cleaning operations: Trim whitespace, standardize text, add sales_agent_id index

CREATE OR REPLACE TABLE Legacy_pty.Silver.sales_teams AS
WITH indexed_agents AS (
  SELECT 
    ROW_NUMBER() OVER (ORDER BY regional_office, manager, sales_agent) as sales_agent_id,
    sales_agent,
    manager,
    regional_office
  FROM Legacy_pty.Bronze.sales_teams
  WHERE sales_agent IS NOT NULL
)
SELECT 
  sales_agent_id,
  TRIM(sales_agent) as sales_agent,
  TRIM(manager) as manager,
  TRIM(regional_office) as regional_office,
  current_timestamp() as cleaned_at
FROM indexed_agents;

In [0]:
-- Silver Layer: Cleaned Data Dictionary
-- Data cleaning operations: Trim whitespace, standardize text

CREATE OR REPLACE TABLE Legacy_pty.Silver.data_dictionary AS
SELECT 
  TRIM(LOWER(Table)) as table_name,
  TRIM(LOWER(Field)) as field_name,
  TRIM(Description) as description,
  current_timestamp() as cleaned_at
FROM Legacy_pty.Bronze.data_dictionary
WHERE Table IS NOT NULL
  AND Field IS NOT NULL;

In [0]:
-- Summary: Data Cleaning Operations from Bronze to Silver
-- 
-- All 5 tables cleaned and loaded into Legacy_pty.Silver:
--
-- 1. accounts:
--    - Fixed typo: 'technolgy' → 'technology'
--    - Trimmed whitespace from all text fields
--    - Standardized sector names to lowercase
--    - Filtered out NULL account names
--    - Added cleaned_at timestamp
--
-- 2. products:
--    - Trimmed whitespace from product and series names
--    - Filtered out NULL products and invalid prices
--    - Added cleaned_at timestamp
--
-- 3. sales_pipeline:
--    - Trimmed whitespace from all text fields
--    - Calculated days_to_close (close_date - engage_date)
--    - Filtered out NULL opportunity_id, sales_agent, engage_date
--    - Added cleaned_at timestamp
--
-- 4. sales_teams:
--    - Trimmed whitespace from all text fields
--    - Filtered out NULL sales_agent names
--    - Added cleaned_at timestamp
--
-- 5. data_dictionary:
--    - Trimmed whitespace from all fields
--    - Standardized table/field names to lowercase
--    - Filtered out NULL table or field names
--    - Added cleaned_at timestamp
--
-- No aggregations or enrichments performed - data is ready for Gold layer analytics

SELECT 'Data cleaning complete!' as status;

In [0]:
-- SUMMARY RELATIONSHIP REPORT
-- Executive overview of data relationships

SELECT 
  'SCHEMA PATTERN' as metric,
  'Star Schema' as value,
  'sales_pipeline is central fact table' as description

UNION ALL

SELECT 
  'FACT TABLE',
  'sales_pipeline',
  CONCAT((SELECT COUNT(*) FROM Legacy_pty.Silver.sales_pipeline), ' opportunity records')

UNION ALL

SELECT 
  'DIMENSION TABLES',
  '3 tables',
  'accounts (85), products (7), sales_teams (35)'

UNION ALL

SELECT 
  'METADATA TABLE',
  'data_dictionary',
  '17 field definitions'

UNION ALL

SELECT 
  'FOREIGN KEY COUNT',
  '4 relationships',
  '3 standard + 1 self-reference'

UNION ALL

SELECT 
  'REFERENTIAL INTEGRITY',
  '100% Valid',
  'All foreign keys resolved after cleaning'

UNION ALL

SELECT 
  'DATA QUALITY FIXES',
  '2 issues resolved',
  '1. GTXPro → GTX Pro, 2. Added surrogate keys'

UNION ALL

SELECT 
  'PRIMARY KEYS',
  'All tables indexed',
  'account_id, product_id, sales_agent_id, opportunity_id'

UNION ALL

SELECT 
  'HIERARCHICAL DATA',
  'accounts table',
  '7 parent companies with 13 subsidiaries'

UNION ALL

SELECT 
  'JOIN COMPLEXITY',
  'Simple star joins',
  'All joins are on single string columns';


In [0]:
-- FULL RELATIONSHIP REPORT
-- Detailed analysis of all table relationships

WITH relationship_stats AS (
  -- Relationship 1: sales_pipeline → accounts
  SELECT 
    1 as order_num,
    'sales_pipeline → accounts' as relationship,
    'Many-to-One' as cardinality,
    'account' as foreign_key_field,
    COUNT(*) as fact_records,
    COUNT(DISTINCT sp.account) as distinct_fk_values,
    (SELECT COUNT(*) FROM Legacy_pty.Silver.accounts) as dimension_records,
    COUNT(DISTINCT a.account_id) as matched_dimension_records,
    ROUND(100.0 * COUNT(DISTINCT a.account_id) / (SELECT COUNT(*) FROM Legacy_pty.Silver.accounts), 2) as dimension_coverage_pct,
    'All accounts referenced in sales pipeline' as notes
  FROM Legacy_pty.Silver.sales_pipeline sp
  LEFT JOIN Legacy_pty.Silver.accounts a ON sp.account = a.account
  
  UNION ALL
  
  -- Relationship 2: sales_pipeline → products
  SELECT 
    2 as order_num,
    'sales_pipeline → products' as relationship,
    'Many-to-One' as cardinality,
    'product' as foreign_key_field,
    COUNT(*) as fact_records,
    COUNT(DISTINCT sp.product) as distinct_fk_values,
    (SELECT COUNT(*) FROM Legacy_pty.Silver.products) as dimension_records,
    COUNT(DISTINCT p.product_id) as matched_dimension_records,
    ROUND(100.0 * COUNT(DISTINCT p.product_id) / (SELECT COUNT(*) FROM Legacy_pty.Silver.products), 2) as dimension_coverage_pct,
    'All products in catalog are sold' as notes
  FROM Legacy_pty.Silver.sales_pipeline sp
  LEFT JOIN Legacy_pty.Silver.products p ON sp.product = p.product
  
  UNION ALL
  
  -- Relationship 3: sales_pipeline → sales_teams
  SELECT 
    3 as order_num,
    'sales_pipeline → sales_teams' as relationship,
    'Many-to-One' as cardinality,
    'sales_agent' as foreign_key_field,
    COUNT(*) as fact_records,
    COUNT(DISTINCT sp.sales_agent) as distinct_fk_values,
    (SELECT COUNT(*) FROM Legacy_pty.Silver.sales_teams) as dimension_records,
    COUNT(DISTINCT st.sales_agent_id) as matched_dimension_records,
    ROUND(100.0 * COUNT(DISTINCT st.sales_agent_id) / (SELECT COUNT(*) FROM Legacy_pty.Silver.sales_teams), 2) as dimension_coverage_pct,
    '30 of 35 sales agents have sales' as notes
  FROM Legacy_pty.Silver.sales_pipeline sp
  LEFT JOIN Legacy_pty.Silver.sales_teams st ON sp.sales_agent = st.sales_agent
  
  UNION ALL
  
  -- Relationship 4: accounts self-reference
  SELECT 
    4 as order_num,
    'accounts → accounts' as relationship,
    'Many-to-One (Self)' as cardinality,
    'subsidiary_of' as foreign_key_field,
    (SELECT COUNT(*) FROM Legacy_pty.Silver.accounts WHERE subsidiary_of IS NOT NULL) as fact_records,
    COUNT(DISTINCT a1.subsidiary_of) as distinct_fk_values,
    (SELECT COUNT(*) FROM Legacy_pty.Silver.accounts) as dimension_records,
    COUNT(DISTINCT a2.account_id) as matched_dimension_records,
    ROUND(100.0 * COUNT(DISTINCT a2.account_id) / COUNT(DISTINCT a1.subsidiary_of), 2) as dimension_coverage_pct,
    '7 parent companies, 13 subsidiaries' as notes
  FROM Legacy_pty.Silver.accounts a1
  LEFT JOIN Legacy_pty.Silver.accounts a2 ON a1.subsidiary_of = a2.account
  WHERE a1.subsidiary_of IS NOT NULL
)
SELECT 
  relationship,
  cardinality,
  foreign_key_field,
  fact_records,
  distinct_fk_values,
  dimension_records,
  matched_dimension_records,
  dimension_coverage_pct,
  notes
FROM relationship_stats
ORDER BY order_num;

In [0]:
-- Relationship Diagram: Star Schema Structure
-- 
-- DIMENSION TABLES:
--
-- ┌────────────────────────────────┐
-- │   accounts (DIMENSION)      │
-- │   - account_id (PK)         │
-- │   - account (UNIQUE)        │
-- │   - sector                  │
-- │   - year_established        │
-- │   - revenue                 │
-- │   - employees               │
-- │   - office_location         │
-- │   - subsidiary_of (FK) ╭───┐
-- │   85 records                │   │
-- └────────────────────────────────┘   │
--          ▲                        │
--          │                        │
--    account (FK)          subsidiary_of
--          │                   (self-ref)
--          │                        │
--          │                        │
-- ┌────────────────────────────────┐       │
-- │  products (DIMENSION)      │       │
-- │  - product_id (PK)         │       │
-- │  - product (UNIQUE)        │       │
-- │  - series                  │       │
-- │  - sales_price             │       │
-- │  7 records                 │       │
-- └────────────────────────────────┘       │
--          ▲                        │
--          │                        │
--    product (FK)                   │
--          │                        │
--          │                        │
--          │                        │
--  ┌───────┤                        │
--  │       │                        │
--  │ ┌──────────────────────────────────────────┐ │
--  │ │  sales_pipeline (FACT TABLE)        │ │
--  │ │  - opportunity_id (PK)               │ │
--  │ │  - sales_agent (FK)                  │ │
--  │ │  - product (FK)                      ├─┘
--  │ │  - account (FK)                      │
--  └─├─ - deal_stage                         │
--    │  - engage_date                        │
--    │  - close_date                         │
--    │  - close_value                        │
--    │  - days_to_close (CALCULATED)         │
--    │  8,800+ records                       │
--    │ └──────────────────────────────────────────┘
--    │
--    │ sales_agent (FK)
--    │
--    ▼
-- ┌────────────────────────────────┐
-- │ sales_teams (DIMENSION)     │
-- │ - sales_agent_id (PK)       │
-- │ - sales_agent (UNIQUE)      │
-- │ - manager                   │
-- │ - regional_office           │
-- │ 35 records                  │
-- └────────────────────────────────┘
--
-- METADATA TABLE (standalone):
-- ┌────────────────────────────────┐
-- │ data_dictionary (METADATA)  │
-- │ - table_name                │
-- │ - field_name                │
-- │ - description               │
-- │ 17 records                  │
-- └────────────────────────────────┘

SELECT 'Relationship diagram displayed above' as status;

In [0]:
-- Foreign Key Validation: Referential Integrity Check
-- Validates all relationships between tables
-- Business Purpose: Data quality assurance - ensures all joins will succeed without orphan records
-- Expected Result: 100% match_percentage for all relationships (indicates clean data)

WITH validation_results AS (
  -- =========================================================================
  -- CHECK 1: sales_pipeline → accounts (CRITICAL: Every deal must have a customer)
  -- =========================================================================
  SELECT 
    'sales_pipeline.account → accounts.account' as foreign_key,
    COUNT(DISTINCT sp.account) as total_values,
    COUNT(DISTINCT CASE WHEN a.account IS NOT NULL THEN sp.account END) as matched_values,
    COUNT(DISTINCT sp.account) - COUNT(DISTINCT CASE WHEN a.account IS NOT NULL THEN sp.account END) as unmatched_values,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.account IS NOT NULL THEN sp.account END) / COUNT(DISTINCT sp.account), 2) as match_percentage
  FROM Legacy_pty.Silver.sales_pipeline sp
  LEFT JOIN Legacy_pty.Silver.accounts a ON sp.account = a.account
  
  UNION ALL
  
  -- =========================================================================
  -- CHECK 2: sales_pipeline → products (CRITICAL: Every deal must have a product)
  -- =========================================================================
  SELECT 
    'sales_pipeline.product → products.product' as foreign_key,
    COUNT(DISTINCT sp.product) as total_values,
    COUNT(DISTINCT CASE WHEN p.product IS NOT NULL THEN sp.product END) as matched_values,
    COUNT(DISTINCT sp.product) - COUNT(DISTINCT CASE WHEN p.product IS NOT NULL THEN sp.product END) as unmatched_values,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN p.product IS NOT NULL THEN sp.product END) / COUNT(DISTINCT sp.product), 2) as match_percentage
  FROM Legacy_pty.Silver.sales_pipeline sp
  LEFT JOIN Legacy_pty.Silver.products p ON sp.product = p.product
  
  UNION ALL
  
  -- =========================================================================
  -- CHECK 3: sales_pipeline → sales_teams (CRITICAL: Every deal must have an owner)
  -- =========================================================================
  SELECT 
    'sales_pipeline.sales_agent → sales_teams.sales_agent' as foreign_key,
    COUNT(DISTINCT sp.sales_agent) as total_values,
    COUNT(DISTINCT CASE WHEN st.sales_agent IS NOT NULL THEN sp.sales_agent END) as matched_values,
    COUNT(DISTINCT sp.sales_agent) - COUNT(DISTINCT CASE WHEN st.sales_agent IS NOT NULL THEN sp.sales_agent END) as unmatched_values,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN st.sales_agent IS NOT NULL THEN sp.sales_agent END) / COUNT(DISTINCT sp.sales_agent), 2) as match_percentage
  FROM Legacy_pty.Silver.sales_pipeline sp
  LEFT JOIN Legacy_pty.Silver.sales_teams st ON sp.sales_agent = st.sales_agent
  
  UNION ALL
  
  -- =========================================================================
  -- CHECK 4: accounts → accounts (OPTIONAL: Parent-child hierarchy)
  -- =========================================================================
  -- Self-referencing foreign key: subsidiary_of points to parent account
  -- Some unmatched values are OK here (parent companies have NULL subsidiary_of)
  SELECT 
    'accounts.subsidiary_of → accounts.account (hierarchy)' as foreign_key,
    COUNT(DISTINCT a1.subsidiary_of) as total_values,
    COUNT(DISTINCT CASE WHEN a2.account IS NOT NULL THEN a1.subsidiary_of END) as matched_values,
    COUNT(DISTINCT a1.subsidiary_of) - COUNT(DISTINCT CASE WHEN a2.account IS NOT NULL THEN a1.subsidiary_of END) as unmatched_values,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN a2.account IS NOT NULL THEN a1.subsidiary_of END) / COUNT(DISTINCT a1.subsidiary_of), 2) as match_percentage
  FROM Legacy_pty.Silver.accounts a1
  LEFT JOIN Legacy_pty.Silver.accounts a2 ON a1.subsidiary_of = a2.account
  WHERE a1.subsidiary_of IS NOT NULL  -- Only validate non-NULL subsidiary_of values
)
SELECT 
  foreign_key,
  total_values,
  matched_values,
  unmatched_values,
  match_percentage,
  -- DATA QUALITY STATUS:
  -- 100% = Perfect referential integrity (all joins will succeed)
  -- <100% = Data quality issue (orphan records exist, investigate immediately)
  CASE 
    WHEN match_percentage = 100 THEN '✓ Valid'
    WHEN match_percentage >= 95 THEN '⚠ Warning: Minor issues'
    ELSE '✗ Critical: Data quality failure'
  END as validation_status
FROM validation_results
ORDER BY match_percentage ASC;  -- Show problems first

-- INTERPRETATION:
-- If any relationship shows <100% match:
-- 1. Check for typos in foreign key values (e.g., 'GTXPro' vs 'GTX Pro')
-- 2. Verify Bronze → Silver cleaning logic
-- 3. Investigate whether missing dimension records should exist
-- 4. Fix data before creating Gold layer aggregates

In [0]:
-- EXAMPLE: Cross-Dimensional Analysis Using Gold Layer Tables
-- Practical queries showing how to join Gold tables for insights

-- =============================================================================
-- EXAMPLE 1: Product Performance by Customer Sector
-- Combines: product_sales_performance + account_sales_performance
-- =============================================================================

WITH product_account_detail AS (
  SELECT 
    sp.product,
    a.sector,
    a.account,
    sp.deal_stage,
    sp.close_value
  FROM Legacy_pty.Silver.sales_pipeline sp
  JOIN Legacy_pty.Silver.accounts a ON sp.account = a.account
)
SELECT 
  p.product,
  p.series,
  pad.sector,
  COUNT(*) as opportunities,
  COUNT(CASE WHEN pad.deal_stage = 'Won' THEN 1 END) as won_deals,
  SUM(CASE WHEN pad.deal_stage = 'Won' THEN pad.close_value ELSE 0 END) as sector_revenue,
  p.total_revenue as overall_product_revenue,
  ROUND(100.0 * SUM(CASE WHEN pad.deal_stage = 'Won' THEN pad.close_value ELSE 0 END) / p.total_revenue, 2) as pct_of_product_revenue
FROM Legacy_pty.Gold.product_sales_performance p
JOIN product_account_detail pad ON p.product = pad.product
GROUP BY p.product, p.series, pad.sector, p.total_revenue
ORDER BY p.product, sector_revenue DESC;


-- =============================================================================
-- EXAMPLE 2: Sales Agent Performance by Regional Office
-- Combines: sales_agent_performance (hierarchical aggregation)
-- =============================================================================

SELECT 
  regional_office,
  manager,
  COUNT(*) as team_size,
  COUNT(CASE WHEN performance_tier = 'Top Performer' THEN 1 END) as top_performers,
  COUNT(CASE WHEN performance_tier = 'High Performer' THEN 1 END) as high_performers,
  COUNT(CASE WHEN performance_tier = 'No Activity' THEN 1 END) as inactive_agents,
  SUM(total_revenue) as team_total_revenue,
  ROUND(AVG(win_rate_pct), 2) as avg_team_win_rate,
  ROUND(AVG(avg_deal_value), 2) as avg_team_deal_value
FROM Legacy_pty.Gold.sales_agent_performance
GROUP BY regional_office, manager
ORDER BY team_total_revenue DESC;


-- =============================================================================
-- EXAMPLE 3: Account Hierarchy Roll-Up (Parent Company Performance)
-- Combines: account_sales_performance (self-referencing hierarchy)
-- =============================================================================

WITH account_hierarchy AS (
  SELECT 
    account,
    account_id,
    subsidiary_of,
    total_sales_revenue as direct_revenue,
    total_opportunities as direct_opportunities,
    customer_tier
  FROM Legacy_pty.Gold.account_sales_performance
),
parent_rollup AS (
  SELECT 
    parent.account as parent_company,
    parent.account_id as parent_account_id,
    parent.direct_revenue as parent_direct_revenue,
    COUNT(sub.account) as num_subsidiaries,
    SUM(sub.direct_revenue) as subsidiary_total_revenue,
    parent.direct_revenue + COALESCE(SUM(sub.direct_revenue), 0) as consolidated_revenue
  FROM account_hierarchy parent
  LEFT JOIN account_hierarchy sub ON parent.account = sub.subsidiary_of
  WHERE parent.subsidiary_of IS NULL  -- Only parent companies
  GROUP BY parent.account, parent.account_id, parent.direct_revenue
)
SELECT 
  parent_company,
  num_subsidiaries,
  parent_direct_revenue,
  subsidiary_total_revenue,
  consolidated_revenue,
  ROUND(100.0 * subsidiary_total_revenue / NULLIF(consolidated_revenue, 0), 2) as subsidiary_contribution_pct
FROM parent_rollup
ORDER BY consolidated_revenue DESC;


-- =============================================================================
-- EXAMPLE 4: Product Performance Trends Over Time
-- Combines: monthly_sales_trends + product_sales_performance
-- =============================================================================

WITH monthly_product_detail AS (
  SELECT 
    YEAR(close_date) as year,
    MONTH(close_date) as month,
    product,
    COUNT(*) as monthly_opportunities,
    SUM(CASE WHEN deal_stage = 'Won' THEN close_value ELSE 0 END) as monthly_revenue
  FROM Legacy_pty.Silver.sales_pipeline
  WHERE close_date IS NOT NULL
  GROUP BY YEAR(close_date), MONTH(close_date), product
)
SELECT 
  mpd.year,
  mpd.month,
  mpd.product,
  p.series,
  mpd.monthly_revenue,
  p.total_revenue as product_lifetime_revenue,
  ROUND(100.0 * mpd.monthly_revenue / p.total_revenue, 2) as pct_of_lifetime_revenue,
  mst.total_revenue as market_monthly_revenue,
  ROUND(100.0 * mpd.monthly_revenue / mst.total_revenue, 2) as market_share_pct
FROM monthly_product_detail mpd
JOIN Legacy_pty.Gold.product_sales_performance p ON mpd.product = p.product
JOIN Legacy_pty.Gold.monthly_sales_trends mst ON mpd.year = mst.year AND mpd.month = mst.month
ORDER BY mpd.year DESC, mpd.month DESC, mpd.monthly_revenue DESC;

In [0]:
-- GOLD LAYER RELATIONSHIP DIAGRAM
-- Shows conformed dimensions and cross-dimensional analysis patterns

SELECT '
╔══════════════════════════════════════════════════════════════════════════════════════════╗
║                            GOLD LAYER ARCHITECTURE                                       ║
║                        Analytical Aggregates with Conformed Dimensions                   ║
╚══════════════════════════════════════════════════════════════════════════════════════════╝

┌─────────────────────────────────────────────────────────────────────────────────────────┐
│                              DIMENSION AGGREGATES (Grain: Entity)                       │
├─────────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                          │
│  ┌────────────────────────────┐     ┌────────────────────────────┐                    │
│  │  product_sales_performance│     │ account_sales_performance  │                    │
│  ├────────────────────────────┤     ├────────────────────────────┤                    │
│  │ • product_id (PK)         │     │ • account_id (PK)          │                    │
│  │ • product                 │     │ • account                  │                    │
│  │ • series                  │     │ • sector                   │                    │
│  │ • total_opportunities     │     │ • subsidiary_of  ─────┐    │  SELF-REFERENCING  │
│  │ • won/lost/active_deals   │     │ • total_opportunities  │    │  HIERARCHY         │
│  │ • win_rate_pct           │     │ • won/lost deals       │    │  (Parent-Child)    │
│  │ • total_revenue          │     │ • customer_tier        └────┤                    │
│  │ • avg_deal_value         │     │ • total_sales_revenue      │                    │
│  └────────────────────────────┘     └────────────────────────────┘                    │
│           7 products                        85 accounts                                 │
│                                                                                          │
│                     ┌────────────────────────────┐                                     │
│                     │ sales_agent_performance    │                                     │
│                     ├────────────────────────────┤                                     │
│                     │ • sales_agent_id (PK)     │                                     │
│                     │ • sales_agent             │                                     │
│                     │ • manager          ───┐   │  ORGANIZATIONAL                     │
│                     │ • regional_office  ───┤   │  HIERARCHY                          │
│                     │ • total_opportunities │   │  (Manager/Office)                   │
│                     │ • won/lost deals      │   │                                     │
│                     │ • performance_tier    │   │                                     │
│                     │ • total_revenue       │   │                                     │
│                     └────────────────────────┘   │                                     │
│                          35 sales agents         │                                     │
└──────────────────────────────────────────────────┴─────────────────────────────────────┘
                                      │
                                      │ Conformed Dimensions
                                      │ (Join through Silver.sales_pipeline)
                                      ▼
┌─────────────────────────────────────────────────────────────────────────────────────────┐
│                          TIME-SERIES FACT (Grain: Month)                                │
├─────────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                          │
│                        ┌────────────────────────────────────┐                          │
│                        │    monthly_sales_trends            │                          │
│                        ├────────────────────────────────────┤                          │
│                        │ • year (PK)                        │                          │
│                        │ • month (PK)                       │                          │
│                        │ • month_start_date                 │                          │
│                        │ • total_opportunities              │                          │
│                        │ • won/lost deals                   │                          │
│                        │ • win_rate_pct                     │                          │
│                        │ • total_revenue                    │                          │
│                        │ • unique_accounts                  │                          │
│                        │ • unique_products                  │                          │
│                        │ • active_sales_agents              │                          │
│                        └────────────────────────────────────┘                          │
│                               ~60 months (2017-2022)                                    │
│                                                                                          │
└─────────────────────────────────────────────────────────────────────────────────────────┘

═══════════════════════════════════════════════════════════════════════════════════════════
  RELATIONSHIP PATTERNS:
═══════════════════════════════════════════════════════════════════════════════════════════

  1. CONFORMED DIMENSIONS (Many-to-Many Cross-Analysis)
     → Product × Account: Which products sell to which customer segments?
     → Product × Agent: Which reps excel at selling which products?
     → Account × Agent: Which reps manage which accounts?
     → All × Time: Trend analysis across any dimension
     
     JOIN METHOD: Through Silver.sales_pipeline fact table
     
  2. HIERARCHICAL DIMENSIONS
     → Account Hierarchy: subsidiary_of enables parent company roll-ups
     → Sales Team Hierarchy: manager/regional_office enables team aggregation
     
  3. DRILL-DOWN CAPABILITY  
     → Any Gold aggregate can drill back to Silver.sales_pipeline for detail
     → Maintains referential integrity through conformed dimension keys

═══════════════════════════════════════════════════════════════════════════════════════════
' as relationship_diagram;

In [0]:
-- GOLD LAYER: Cross-Dimensional Relationship Patterns
-- How to join Gold tables together for multi-dimensional analysis

-- =============================================================================
-- RELATIONSHIP PATTERN 1: Drill-Back to Silver (Conformed Dimensions)
-- =============================================================================
-- Each Gold table can be joined back to Silver layer for detailed drill-down:
--
-- product_sales_performance.product → Silver.sales_pipeline.product → detailed opportunities
-- account_sales_performance.account → Silver.sales_pipeline.account → detailed opportunities  
-- sales_agent_performance.sales_agent → Silver.sales_pipeline.sales_agent → detailed opportunities
-- monthly_sales_trends.(year,month) → Silver.sales_pipeline.close_date → detailed opportunities
--

-- =============================================================================
-- RELATIONSHIP PATTERN 2: Hierarchical Dimensions
-- =============================================================================
--
-- account_sales_performance has SELF-REFERENCING hierarchy:
--   └─ subsidiary_of field creates parent-child relationships
--   └─ Use recursive CTE to roll up subsidiary performance to parent companies
--
-- sales_agent_performance has ORGANIZATIONAL hierarchy:
--   └─ manager field groups reps by supervisor
--   └─ regional_office field groups reps by location
--   └─ Use GROUP BY to aggregate performance by manager or office
--

-- =============================================================================
-- RELATIONSHIP PATTERN 3: Many-to-Many Cross-Dimensional Analysis
-- =============================================================================
-- Gold tables don't have direct FKs but share common fact source (sales_pipeline)
-- Join through Silver.sales_pipeline to create multi-dimensional views:

SELECT 
  'Product × Account' as relationship_type,
  'product_sales_performance + account_sales_performance' as gold_tables,
  'JOIN through Silver.sales_pipeline using product and account' as join_pattern,
  'Which products perform best with which customer segments?' as use_case

UNION ALL

SELECT 
  'Product × Sales Agent',
  'product_sales_performance + sales_agent_performance',
  'JOIN through Silver.sales_pipeline using product and sales_agent',
  'Which reps are best at selling which products?'

UNION ALL

SELECT 
  'Account × Sales Agent',
  'account_sales_performance + sales_agent_performance',
  'JOIN through Silver.sales_pipeline using account and sales_agent',
  'Which reps manage which accounts most effectively?'

UNION ALL

SELECT 
  'Time × All Dimensions',
  'monthly_sales_trends + any dimension table',
  'JOIN through Silver.sales_pipeline using close_date + dimension key',
  'Trend analysis: How does each product/account/rep perform over time?'

UNION ALL

SELECT 
  'Product × Account × Agent',
  'All three dimension tables',
  'JOIN through Silver.sales_pipeline using all three keys',
  'Full dimensional analysis: product-account-rep combinations';

-- =============================================================================
-- RELATIONSHIP PATTERN 4: Time-Based Relationships
-- =============================================================================
-- monthly_sales_trends can be joined to dimensional tables using date logic:
--   └─ Join sales_agent_performance using first_engagement_date/last_close_date
--   └─ Join account_sales_performance using first_engagement_date/last_close_date  
--   └─ Enables cohort analysis and time-based segmentation

In [0]:
-- Gold Layer: Product Sales Performance
-- Aggregated sales metrics by product
-- Business Purpose: Product portfolio analysis, pricing strategy, inventory decisions
-- Grain: One row per product (7 rows total)

CREATE OR REPLACE TABLE Legacy_pty.Gold.product_sales_performance AS
SELECT 
  p.product_id,
  p.product,
  p.series,      -- GTX Basic, GTX Plus, GTX Pro for series-level rollups
  p.sales_price, -- List price from catalog
  
  -- OPPORTUNITY METRICS: Full pipeline view
  COUNT(sp.opportunity_id) as total_opportunities,
  COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END) as won_deals,
  COUNT(CASE WHEN sp.deal_stage = 'Lost' THEN 1 END) as lost_deals,
  COUNT(CASE WHEN sp.deal_stage NOT IN ('Won', 'Lost') THEN 1 END) as active_deals,
  
  -- WIN RATE: Key performance indicator for product-market fit
  -- High win rate (>60%) = strong demand; Low win rate (<40%) = pricing/positioning issue
  ROUND(100.0 * COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END) / COUNT(sp.opportunity_id), 2) as win_rate_pct,
  
  -- REVENUE METRICS: Only from Won deals
  SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END) as total_revenue,
  ROUND(AVG(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value END), 2) as avg_deal_value,
  MIN(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value END) as min_deal_value,
  MAX(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value END) as max_deal_value,
  
  -- SALES VELOCITY: Average time to close won deals
  -- Lower is better - indicates efficient sales process
  ROUND(AVG(CASE WHEN sp.deal_stage = 'Won' THEN sp.days_to_close END), 1) as avg_days_to_close,
  
  current_timestamp() as created_at
FROM Legacy_pty.Silver.products p
-- LEFT JOIN preserves all products even if they have no sales (important for inventory analysis)
LEFT JOIN Legacy_pty.Silver.sales_pipeline sp ON p.product = sp.product
GROUP BY p.product_id, p.product, p.series, p.sales_price
ORDER BY total_revenue DESC;  -- Most valuable products first

In [0]:
-- Gold Layer: Account Sales Performance
-- Aggregated sales metrics by customer account
-- Business Purpose: Customer relationship health, account segmentation, expansion opportunities
-- Grain: One row per account (85 rows total)

CREATE OR REPLACE TABLE Legacy_pty.Gold.account_sales_performance AS
SELECT 
  a.account_id,
  a.account,
  a.sector,              -- Industry vertical for sector-level analysis
  a.office_location,
  a.year_established,
  a.revenue as company_revenue,  -- Customer's company size (not our sales to them)
  a.employees,
  a.subsidiary_of,       -- Parent company relationship for hierarchy rollups
  
  -- OPPORTUNITY METRICS
  COUNT(sp.opportunity_id) as total_opportunities,
  COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END) as won_deals,
  COUNT(CASE WHEN sp.deal_stage = 'Lost' THEN 1 END) as lost_deals,
  
  -- WIN RATE: Customer relationship quality indicator
  ROUND(100.0 * COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END) / COUNT(sp.opportunity_id), 2) as win_rate_pct,
  
  -- REVENUE: Total value we've generated from this customer
  SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END) as total_sales_revenue,
  ROUND(AVG(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value END), 2) as avg_deal_value,
  
  -- PRODUCT DIVERSITY: Number of unique products purchased
  -- High diversity = strong multi-product relationship
  COUNT(DISTINCT sp.product) as products_purchased,
  
  -- SALES COVERAGE: Number of reps engaged with this account
  -- Multiple reps = complex sale or handoff issues
  COUNT(DISTINCT sp.sales_agent) as sales_agents_engaged,
  
  -- RELATIONSHIP TIMELINE
  MIN(sp.engage_date) as first_engagement_date,  -- Customer acquisition date
  MAX(sp.close_date) as last_close_date,         -- Most recent purchase
  
  -- SALES VELOCITY
  ROUND(AVG(CASE WHEN sp.deal_stage = 'Won' THEN sp.days_to_close END), 1) as avg_days_to_close,
  
  -- CUSTOMER SEGMENTATION: Revenue-based tiering for account management strategy
  -- Enterprise (>$100K): Strategic accounts, dedicated CSM
  -- Large ($50K-$100K): Key accounts, quarterly reviews
  -- Medium ($10K-$50K): Standard support
  -- Small (<$10K): Self-service or inside sales
  CASE 
    WHEN SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END) >= 100000 THEN 'Enterprise'
    WHEN SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END) >= 50000 THEN 'Large'
    WHEN SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END) >= 10000 THEN 'Medium'
    ELSE 'Small'
  END as customer_tier,
  
  current_timestamp() as created_at
FROM Legacy_pty.Silver.accounts a
LEFT JOIN Legacy_pty.Silver.sales_pipeline sp ON a.account = sp.account
GROUP BY a.account_id, a.account, a.sector, a.office_location, a.year_established, a.revenue, a.employees, a.subsidiary_of
ORDER BY total_sales_revenue DESC;  -- Highest value customers first

In [0]:
-- Gold Layer: Sales Agent Performance
-- Individual sales rep performance metrics
-- Business Purpose: Sales team management, quota setting, compensation planning, coaching needs
-- Grain: One row per sales agent (35 rows total)

CREATE OR REPLACE TABLE Legacy_pty.Gold.sales_agent_performance AS
SELECT 
  st.sales_agent_id,
  st.sales_agent,
  st.manager,           -- For team-level rollups and coaching accountability
  st.regional_office,   -- Geographic performance comparison (East, West, Central)
  
  -- ACTIVITY METRICS: Pipeline management
  COUNT(sp.opportunity_id) as total_opportunities,
  COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END) as won_deals,
  COUNT(CASE WHEN sp.deal_stage = 'Lost' THEN 1 END) as lost_deals,
  COUNT(CASE WHEN sp.deal_stage = 'Engaging' THEN 1 END) as engaging_deals,
  COUNT(CASE WHEN sp.deal_stage = 'Prospecting' THEN 1 END) as prospecting_deals,
  
  -- WIN RATE: Primary sales effectiveness metric
  -- Company average ~52%; top performers achieve 55%+
  ROUND(100.0 * COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END) / COUNT(sp.opportunity_id), 2) as win_rate_pct,
  
  -- REVENUE: Total quota attainment (used for commission calculations)
  SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END) as total_revenue,
  ROUND(AVG(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value END), 2) as avg_deal_value,
  
  -- ACCOUNT COVERAGE: Number of unique customers managed
  -- Territory size indicator and account management complexity
  COUNT(DISTINCT sp.account) as unique_accounts,
  
  -- PRODUCT EXPERTISE: Number of unique products sold
  -- High diversity = versatile rep; Low diversity = product specialist
  COUNT(DISTINCT sp.product) as products_sold,
  
  -- SALES VELOCITY: Speed to close
  ROUND(AVG(CASE WHEN sp.deal_stage = 'Won' THEN sp.days_to_close END), 1) as avg_days_to_close,
  
  -- TENURE INDICATORS
  MIN(sp.engage_date) as first_engagement_date,  -- Ramp-up start date
  MAX(sp.close_date) as last_close_date,         -- Last activity (watch for stale reps)
  
  -- PERFORMANCE TIER: Multi-factor segmentation for talent management
  -- Top Performer: $1M+ revenue AND 55%+ win rate (coaching others, retention critical)
  -- High Performer: $500K+ revenue AND 50%+ win rate (solid contributors, promotion candidates)
  -- No Activity: Zero closed deals (new hire ramp, PIP candidate, or attrition)
  -- Developing: Everyone else (coaching focus, skill development)
  CASE 
    WHEN SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END) >= 1000000 
         AND ROUND(100.0 * COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END) / COUNT(sp.opportunity_id), 2) >= 55 
         THEN 'Top Performer'
    WHEN SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END) >= 500000 
         AND ROUND(100.0 * COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END) / COUNT(sp.opportunity_id), 2) >= 50 
         THEN 'High Performer'
    WHEN COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END) = 0 
         THEN 'No Activity'
    ELSE 'Developing'
  END as performance_tier,
  
  current_timestamp() as created_at
FROM Legacy_pty.Silver.sales_teams st
LEFT JOIN Legacy_pty.Silver.sales_pipeline sp ON st.sales_agent = sp.sales_agent
GROUP BY st.sales_agent_id, st.sales_agent, st.manager, st.regional_office
ORDER BY total_revenue DESC;  -- Top revenue generators first

In [0]:
-- Gold Layer: Monthly Sales Trends
-- Time-series analysis of sales performance
-- Business Purpose: Revenue forecasting, seasonality detection, capacity planning, board reporting
-- Grain: One row per month (10 months for 2017: Jan-Oct)

CREATE OR REPLACE TABLE Legacy_pty.Gold.monthly_sales_trends AS
SELECT 
  YEAR(close_date) as year,
  MONTH(close_date) as month,
  DATE_TRUNC('MONTH', close_date) as month_start_date,  -- First day of month for time-series charting
  
  -- VOLUME METRICS: Pipeline flow
  COUNT(opportunity_id) as total_opportunities,
  COUNT(CASE WHEN deal_stage = 'Won' THEN 1 END) as won_deals,
  COUNT(CASE WHEN deal_stage = 'Lost' THEN 1 END) as lost_deals,
  
  -- WIN RATE: Month-over-month trend indicator
  -- Watch for drops below 50% (market shift, competitive pressure, or execution issues)
  ROUND(100.0 * COUNT(CASE WHEN deal_stage = 'Won' THEN 1 END) / COUNT(opportunity_id), 2) as win_rate_pct,
  
  -- REVENUE: Primary business metric for financial reporting
  -- Track month-over-month growth rate (target: 10%+ MoM)
  SUM(CASE WHEN deal_stage = 'Won' THEN close_value ELSE 0 END) as total_revenue,
  ROUND(AVG(CASE WHEN deal_stage = 'Won' THEN close_value END), 2) as avg_deal_value,
  
  -- MARKET COVERAGE: Breadth metrics
  COUNT(DISTINCT account) as unique_accounts,    -- Customer acquisition trend
  COUNT(DISTINCT product) as unique_products,    -- Product portfolio utilization
  COUNT(DISTINCT sales_agent) as active_sales_agents,  -- Team capacity (hiring/attrition indicator)
  
  -- SALES VELOCITY: Efficiency trend
  -- Increasing days_to_close may indicate:
  -- - More complex deals (positive: moving upmarket)
  -- - Sales friction (negative: process breakdown)
  ROUND(AVG(CASE WHEN deal_stage = 'Won' THEN days_to_close END), 1) as avg_days_to_close,
  
  current_timestamp() as created_at
FROM Legacy_pty.Silver.sales_pipeline
WHERE close_date IS NOT NULL  -- Only closed deals (Won or Lost) for accurate monthly attribution
GROUP BY YEAR(close_date), MONTH(close_date), DATE_TRUNC('MONTH', close_date)
ORDER BY year DESC, month DESC;  -- Most recent months first

In [0]:
-- GOLD LAYER RELATIONSHIP ANALYSIS
-- Unlike Silver layer with explicit foreign keys, Gold layer uses conformed dimensions
-- These are analytical aggregates that can be joined for multi-dimensional analysis

-- =============================================================================
-- RELATIONSHIP TYPE: Conformed Dimensions (Star Schema Analytics Pattern)
-- =============================================================================

WITH gold_table_summary AS (
  SELECT 'product_sales_performance' as table_name,
         'Product Dimension' as dimension_type,
         'product_id, product' as grain,
         'Product performance metrics aggregated from sales_pipeline' as description,
         ' 7 products' as expected_rows
  
  UNION ALL
  
  SELECT 'account_sales_performance',
         'Customer Dimension',
         'account_id, account',
         'Customer performance metrics with subsidiary hierarchy',
         '85 accounts'
  
  UNION ALL
  
  SELECT 'sales_agent_performance',
         'Sales Rep Dimension',
         'sales_agent_id, sales_agent',
         'Individual rep performance with manager/office hierarchy',
         '35 sales agents'
  
  UNION ALL
  
  SELECT 'monthly_sales_trends',
         'Time Dimension (Fact)',
         'year, month',
         'Time-series metrics across all dimensions',
         '~60 months (2017-2022)'
)
SELECT 
  table_name,
  dimension_type,
  grain,
  description,
  expected_rows
FROM gold_table_summary
ORDER BY 
  CASE dimension_type 
    WHEN 'Time Dimension (Fact)' THEN 1
    ELSE 2
  END,
  table_name;

In [0]:
-- ACCOUNT SECTOR DISTRIBUTION
-- Analyzing revenue, accounts, and performance metrics by industry sector
-- Business Purpose: Market segmentation, vertical strategy, sales specialization planning
-- Insights: Which industries are most profitable? Where should we invest marketing spend?

WITH sector_metrics AS (
  SELECT 
    sector,
    COUNT(DISTINCT account) as total_accounts,
    SUM(total_sales_revenue) as sector_revenue,
    SUM(total_opportunities) as sector_opportunities,
    SUM(won_deals) as sector_won_deals,
    SUM(lost_deals) as sector_lost_deals,
    -- Sector-level win rate (weighted by all opportunities across accounts)
    ROUND(100.0 * SUM(won_deals) / SUM(total_opportunities), 2) as sector_win_rate_pct,
    ROUND(AVG(avg_deal_value), 2) as avg_deal_size,
    ROUND(AVG(avg_days_to_close), 1) as avg_sales_cycle
  FROM Legacy_pty.Gold.account_sales_performance
  WHERE total_sales_revenue > 0  -- Exclude accounts with no closed deals
  GROUP BY sector
),
market_totals AS (
  -- Calculate company-wide totals for market share percentages
  SELECT 
    SUM(sector_revenue) as total_market_revenue,
    SUM(total_accounts) as total_market_accounts
  FROM sector_metrics
)
SELECT 
  sm.sector,
  sm.total_accounts,
  sm.sector_revenue,
  
  -- MARKET SHARE: Revenue concentration by sector
  -- High concentration (>20%) = vertical expertise opportunity
  -- Low concentration (<5%) = underdeveloped market or low priority
  ROUND(100.0 * sm.sector_revenue / mt.total_market_revenue, 2) as revenue_market_share_pct,
  
  -- ACCOUNT SHARE: Customer base distribution
  ROUND(100.0 * sm.total_accounts / mt.total_market_accounts, 2) as account_market_share_pct,
  
  -- REVENUE PER ACCOUNT: Account value by sector
  -- High value = enterprise focus; Low value = volume play
  ROUND(sm.sector_revenue / sm.total_accounts, 0) as revenue_per_account,
  
  sm.sector_win_rate_pct,
  sm.avg_deal_size,
  sm.avg_sales_cycle,
  
  -- PERFORMANCE RANKING: Sectors ordered by total revenue
  -- Top 3 sectors = strategic focus areas for product development and marketing
  ROW_NUMBER() OVER (ORDER BY sm.sector_revenue DESC) as revenue_rank
FROM sector_metrics sm
CROSS JOIN market_totals mt
ORDER BY sm.sector_revenue DESC;  -- Highest revenue sectors first

-- EXPECTED INSIGHTS:
-- 1. Retail likely leads (18%+ market share) - broad customer base
-- 2. Software has highest win rate (60%+) - strong product-market fit
-- 3. Technology/Finance likely have highest avg deal size - enterprise customers
-- 4. Use revenue_rank to identify top 3 sectors for focused growth investment

In [0]:
-- REGIONAL OFFICE PERFORMANCE COMPARISON
-- Compare Central, East, and West office performance metrics
-- Business Purpose: Territory management, resource allocation, best practice sharing, compensation fairness
-- Insights: Which office is most productive? Should we redistribute territories?

WITH regional_metrics AS (
  SELECT 
    regional_office,
    COUNT(DISTINCT sales_agent) as total_agents,
    COUNT(DISTINCT manager) as total_managers,
    SUM(total_revenue) as office_revenue,
    SUM(total_opportunities) as office_opportunities,
    SUM(won_deals) as office_won_deals,
    -- Office-level win rate (all reps combined)
    ROUND(100.0 * SUM(won_deals) / SUM(total_opportunities), 2) as office_win_rate_pct,
    ROUND(AVG(avg_deal_value), 2) as avg_deal_size,
    -- Count top performers per office (succession planning, retention risk)
    COUNT(CASE WHEN performance_tier = 'Top Performer' THEN 1 END) as top_performers_count
  FROM Legacy_pty.Gold.sales_agent_performance
  GROUP BY regional_office
),
company_totals AS (
  -- Calculate company-wide benchmarks
  SELECT 
    SUM(office_revenue) as total_company_revenue,
    SUM(total_agents) as total_company_agents
  FROM regional_metrics
)
SELECT 
  rm.regional_office,
  rm.total_agents,
  rm.total_managers,
  rm.office_revenue,
  
  -- REVENUE CONTRIBUTION: Office share of company revenue
  -- Target: Roughly equal split (33% each) unless territory sizes differ
  ROUND(100.0 * rm.office_revenue / ct.total_company_revenue, 2) as revenue_contribution_pct,
  
  -- PRODUCTIVITY PER REP: Key efficiency metric
  -- Identifies high-performing office models to replicate
  -- Expected: West office leads at $357K/agent (benchmark for others)
  ROUND(rm.office_revenue / rm.total_agents, 0) as revenue_per_agent,
  
  -- TEAM SIZE SHARE
  ROUND(100.0 * rm.total_agents / ct.total_company_agents, 2) as agent_share_pct,
  
  rm.office_win_rate_pct,
  rm.avg_deal_size,
  rm.top_performers_count,
  
  -- PERFORMANCE RANKING: Offices ordered by total revenue
  -- Use for bonus pool allocation and territory expansion decisions
  ROW_NUMBER() OVER (ORDER BY rm.office_revenue DESC) as revenue_rank,
  
  -- EFFICIENCY RANKING: Offices ordered by per-rep productivity
  -- Identifies best practices to scale across company
  ROW_NUMBER() OVER (ORDER BY rm.office_revenue / rm.total_agents DESC) as productivity_rank
FROM regional_metrics rm
CROSS JOIN company_totals ct
ORDER BY rm.office_revenue DESC;

-- EXPECTED INSIGHTS:
-- 1. West: Highest revenue ($3.57M, 35.67%) + highest productivity ($357K/agent)
--    → Replicate their playbook (territory assignment, coaching, sales methodology)
-- 2. Central: Highest win rate (54.10%) + houses 1 Top Performer (Darcel Schlecht)
--    → Document conversion tactics and use for company-wide training
-- 3. East: Largest avg deal size ($2,647) despite lower volume
--    → Enterprise sales expertise - deploy for high-value account coaching

In [0]:
-- SUBSIDIARY-PARENT RELATIONSHIP HIERARCHY
-- Show parent companies with their subsidiaries and consolidated revenue

WITH parent_companies AS (
  SELECT 
    account_id,
    account as parent_company,
    sector,
    office_location,
    company_revenue as parent_company_revenue,
    total_sales_revenue as parent_direct_sales,
    total_opportunities as parent_opportunities,
    won_deals as parent_won_deals,
    win_rate_pct as parent_win_rate,
    customer_tier as parent_tier
  FROM Legacy_pty.Gold.account_sales_performance
  WHERE subsidiary_of IS NULL
),
subsidiary_details AS (
  SELECT 
    subsidiary_of,
    account as subsidiary_name,
    sector as subsidiary_sector,
    office_location as subsidiary_location,
    company_revenue as subsidiary_company_revenue,
    total_sales_revenue as subsidiary_sales,
    total_opportunities as subsidiary_opportunities,
    won_deals as subsidiary_won_deals,
    win_rate_pct as subsidiary_win_rate
  FROM Legacy_pty.Gold.account_sales_performance
  WHERE subsidiary_of IS NOT NULL
),
subsidiary_aggregates AS (
  SELECT 
    subsidiary_of,
    COUNT(*) as num_subsidiaries,
    SUM(subsidiary_sales) as total_subsidiary_sales,
    SUM(subsidiary_opportunities) as total_subsidiary_opportunities,
    SUM(subsidiary_won_deals) as total_subsidiary_won_deals,
    ROUND(AVG(subsidiary_win_rate), 2) as avg_subsidiary_win_rate
  FROM subsidiary_details
  GROUP BY subsidiary_of
),
consolidated_view AS (
  SELECT 
    p.parent_company,
    p.sector,
    p.office_location,
    COALESCE(sa.num_subsidiaries, 0) as num_subsidiaries,
    p.parent_direct_sales,
    COALESCE(sa.total_subsidiary_sales, 0) as subsidiary_sales,
    p.parent_direct_sales + COALESCE(sa.total_subsidiary_sales, 0) as consolidated_revenue,
    p.parent_opportunities,
    COALESCE(sa.total_subsidiary_opportunities, 0) as subsidiary_opportunities,
    p.parent_opportunities + COALESCE(sa.total_subsidiary_opportunities, 0) as consolidated_opportunities,
    p.parent_won_deals,
    COALESCE(sa.total_subsidiary_won_deals, 0) as subsidiary_won_deals,
    p.parent_won_deals + COALESCE(sa.total_subsidiary_won_deals, 0) as consolidated_won_deals,
    p.parent_win_rate,
    COALESCE(sa.avg_subsidiary_win_rate, 0) as avg_subsidiary_win_rate,
    ROUND(100.0 * COALESCE(sa.total_subsidiary_sales, 0) / 
          NULLIF(p.parent_direct_sales + COALESCE(sa.total_subsidiary_sales, 0), 0), 2) as subsidiary_contribution_pct
  FROM parent_companies p
  LEFT JOIN subsidiary_aggregates sa ON p.parent_company = sa.subsidiary_of
)
SELECT 
  parent_company,
  sector,
  office_location,
  num_subsidiaries,
  parent_direct_sales,
  subsidiary_sales,
  consolidated_revenue,
  subsidiary_contribution_pct,
  consolidated_opportunities,
  consolidated_won_deals,
  parent_win_rate,
  avg_subsidiary_win_rate,
  CASE 
    WHEN num_subsidiaries = 0 THEN 'Independent'
    WHEN num_subsidiaries >= 3 THEN 'Major Conglomerate'
    ELSE 'Small Conglomerate'
  END as company_structure
FROM consolidated_view
ORDER BY consolidated_revenue DESC;

In [0]:
-- DETAILED SUBSIDIARY BREAKDOWN
-- Shows which subsidiaries belong to which parent companies

SELECT 
  subsidiary_of as parent_company,
  account as subsidiary_name,
  sector as subsidiary_sector,
  office_location as subsidiary_location,
  total_sales_revenue as subsidiary_revenue,
  total_opportunities,
  won_deals,
  win_rate_pct,
  customer_tier,
  RANK() OVER (PARTITION BY subsidiary_of ORDER BY total_sales_revenue DESC) as rank_within_parent
FROM Legacy_pty.Gold.account_sales_performance
WHERE subsidiary_of IS NOT NULL
ORDER BY subsidiary_of, total_sales_revenue DESC;

In [0]:
-- TOP PERFORMING ACCOUNTS
-- Analyzing customer performance by revenue, engagement, and customer tier

SELECT 
  account,
  sector,
  customer_tier,
  office_location,
  company_revenue,
  total_sales_revenue,
  total_opportunities,
  won_deals,
  lost_deals,
  win_rate_pct,
  avg_deal_value,
  products_purchased,
  sales_agents_engaged,
  avg_days_to_close,
  subsidiary_of,
  CASE 
    WHEN subsidiary_of IS NULL THEN 'Parent Company'
    ELSE 'Subsidiary'
  END as company_type
FROM Legacy_pty.Gold.account_sales_performance
ORDER BY total_sales_revenue DESC
LIMIT 20;

In [0]:
-- SALES TEAM PERFORMANCE ANALYSIS
-- Individual rep performance with manager hierarchy and regional breakdown

SELECT 
  sales_agent,
  manager,
  regional_office,
  performance_tier,
  total_revenue,
  total_opportunities,
  won_deals,
  lost_deals,
  engaging_deals,
  prospecting_deals,
  win_rate_pct,
  avg_deal_value,
  unique_accounts,
  products_sold,
  avg_days_to_close,
  DATEDIFF(CURRENT_DATE(), first_engagement_date) as days_since_first_sale,
  CASE 
    WHEN won_deals = 0 THEN 'No Wins'
    WHEN won_deals < 100 THEN 'Building'
    WHEN won_deals < 200 THEN 'Established'
    ELSE 'Veteran'
  END as experience_level
FROM Legacy_pty.Gold.sales_agent_performance
WHERE total_opportunities > 0
ORDER BY total_revenue DESC;

In [0]:
-- MONTHLY REVENUE TRENDS
-- Time-series analysis showing revenue trajectory and key metrics over time

SELECT 
  year,
  month,
  month_start_date,
  total_revenue,
  total_opportunities,
  won_deals,
  lost_deals,
  win_rate_pct,
  avg_deal_value,
  unique_accounts,
  unique_products,
  active_sales_agents,
  avg_days_to_close,
  -- Calculate month-over-month revenue change
  total_revenue - LAG(total_revenue) OVER (ORDER BY year, month) as revenue_change_mom,
  ROUND(100.0 * (total_revenue - LAG(total_revenue) OVER (ORDER BY year, month)) / 
        NULLIF(LAG(total_revenue) OVER (ORDER BY year, month), 0), 2) as revenue_change_pct_mom,
  -- Calculate 3-month moving average
  ROUND(AVG(total_revenue) OVER (ORDER BY year, month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) as revenue_3mo_avg
FROM Legacy_pty.Gold.monthly_sales_trends
ORDER BY year DESC, month DESC;